PRACTICA 2

In [ ]:
import re
from typing import Optional


class Time:
    """
    Clase que representa una hora en formato AM/PM o 24 horas.
    """

    # Atributos de clase
    TIME_FORMATS: tuple[str, ...] = ("AM", "PM", "24 HOURS")
    time_count: int = 0  # Cuenta el número de objetos Time creados

    # Justificación: Método de instancia (constructor). Necesita 'self' para inicializar
    # los atributos privados del nuevo objeto que se está creando.
    def __init__(self) -> None:
        """
        Inicializa los atributos a 0 (y formato por defecto a "24 HOURS")
        e incrementa el contador de objetos creados.
        """
        self.__hours: int = 0
        self.__minutes: int = 0
        self.__seconds: int = 0
        self.__time_format: str = "24 HOURS"
        Time.time_count += 1

    # ==================== PROPERTIES (Getters) ====================
    # Los atributos privados se exponen mediante @property como pide el enunciado.

    @property
    def hours(self) -> int:
        """Devuelve las horas del objeto."""
        return self.__hours

    @property
    def minutes(self) -> int:
        """Devuelve los minutos del objeto."""
        return self.__minutes

    @property
    def seconds(self) -> int:
        """Devuelve los segundos del objeto."""
        return self.__seconds

    @property
    def time_format(self) -> str:
        """Devuelve el formato de hora del objeto."""
        return self.__time_format

    def __assign_format(self, time_format: str) -> bool:
        """
        Comprueba que time_format tiene un valor correcto y lo asigna al atributo correspondiente.
        Lo convierte a mayúsculas para evitar problemas de capitalización.

        Args:
            time_format: cadena con el formato de hora ("AM", "PM" o "24 HOURS").
        Devuelve:
            True si el formato es correcto, False en caso contrario.
        """
        formatted_str: str = time_format.strip().upper()
        if Time.is_valid_format(formatted_str):
            self.__time_format = formatted_str
            return True
        return False

    def __is_24hour_format(self) -> bool:
        """
        Comprueba si el formato de la hora del objeto es "24 HOURS".

        Devuelve:
            True si el formato es "24 HOURS", False en caso contrario.
        """
        return self.__time_format == "24 HOURS"

   
    def _is_valid_time(self, hours: int, minutes: int, seconds: int) -> bool:
        """
        Comprueba si la hora es correcta según el formato actual del objeto.

        Args:
            hours: horas a validar.
            minutes: minutos a validar.
            seconds: segundos a validar.
        Devuelve:
            True si la hora es correcta, False en caso contrario.
        """
        if not (0 <= minutes <= 59 and 0 <= seconds <= 59):
            return False

        if self.__is_24hour_format():
            return 0 <= hours <= 23
        else:
            return 1 <= hours <= 12

    
    def set_time(self, hours: int, minutes: int, seconds: int, time_format: str) -> bool:
        """
        Asigna una hora al objeto previa validación del formato y de los rangos.

        Args:
            hours: horas (1 a 12 en AM/PM, 0 a 23 en 24 HOURS).
            minutes: minutos (0 a 59).
            seconds: segundos (0 a 59).
            time_format: formato ("AM", "PM" o "24 HOURS").
        Devuelve:
            True si la hora se pudo asignar correctamente, False en caso contrario.
        """
        old_format: str = self.__time_format
        if not self.__assign_format(time_format):
            return False

        if self._is_valid_time(hours, minutes, seconds):
            self.__hours = hours
            self.__minutes = minutes
            self.__seconds = seconds
            return True
        else:
            # Si los números de la hora fallan, restauramos el formato que tenía antes
            self.__time_format = old_format
            return False

    def get_time(self) -> str:
        """
        Devuelve la hora actual del objeto en una cadena formateada.

        Devuelve:
            Cadena con formato "HH:MM:SS FORMATO".
        """
        return f"{self.__hours:02d}:{self.__minutes:02d}:{self.__seconds:02d} {self.__time_format}"

    @classmethod
    def from_string(cls, time_string: str) -> Optional["Time"]:
        """
        Crea una nueva instancia de Time a partir de una cadena.

        Args:
            time_string: cadena con la hora en el formato "HH:MM:SS FORMATO",
                         donde FORMATO es AM, PM o 24 HOURS.
        Devuelve:
            Una instancia de Time con la hora leída, o None si la cadena no es válida.
        """
        
        pattern: str = r"^(\d{1,2}):(\d{1,2}):(\d{1,2})\s+(AM|PM|24 HOURS)$"
        match = re.match(pattern, time_string.strip(), re.IGNORECASE)

        if match:
            hours_str, minutes_str, seconds_str, fmt_str = match.groups()
            new_time = cls()
            if new_time.set_time(int(hours_str), int(minutes_str), int(seconds_str), fmt_str):
                return new_time
            else:
                # Si la hora numérica está fuera de rango, descontamos el objeto temporal fallido
                cls.time_count -= 1
                print("Error: Los valores numéricos de la hora están fuera de rango para ese formato.")
                return None
        else:
            print("Error: El formato de la cadena no es válido. Use 'HH:MM:SS FORMATO' (ej. '14:30:00 24 HOURS').")
            return None

    @staticmethod
    def is_valid_format(time_format: str) -> bool:
        """
        Comprueba si un formato de hora dado es válido.

        Args:
            time_format: cadena con el formato de hora a comprobar.
        Devuelve:
            True si el formato es válido (AM, PM o 24 HOURS), False en caso contrario.
        """
        if not isinstance(time_format, str):
            return False
        return time_format.strip().upper() in Time.TIME_FORMATS

   
    @classmethod
    def get_time_count(cls) -> int:
        """
        Devuelve el número de objetos de la clase que se han creado.

        Devuelve:
            Un número entero de objetos Time creados.
        """
        return cls.time_count




def display_formatted_time(time_obj: Optional[Time]) -> None:
    """
    Función fuera de la clase para mostrar la hora en una cadena con formato.

    Args:
        time_obj: Instancia de la clase Time que se desea mostrar.
    """
    if time_obj is None:
        print("Todavía no se ha establecido ninguna hora válida.")
    else:
        print(f"Hora actual: {time_obj.get_time()} (Total objetos Time creados: {Time.get_time_count()})")


def main() -> None:
    """
    Función principal que ejecuta el programa controlado por menú.
    """
    current_time: Optional[Time] = None

    while True:
        print("\n--- MENÚ DE GESTIÓN DE HORA ---")
        print("1. Introducir una nueva hora")
        print("2. Visualizar hora")
        print("3. Crear una hora a partir de una cadena (\"HH:MM:SS FORMATO\")")
        print("4. Terminar")

        option: str = input("Seleccione una opción (1-4): ").strip()

        if option == "1":
            try:
                hours: int = int(input("Introduzca las horas: "))
                minutes: int = int(input("Introduzca los minutos (0-59): "))
                seconds: int = int(input("Introduzca los segundos (0-59): "))
                time_format: str = input("Introduzca el formato (AM, PM o 24 HOURS): ").strip()

                if not Time.is_valid_format(time_format):
                    print("Error: Formato de hora no válido. Debe ser AM, PM o 24 HOURS.")
                    continue

                if current_time is None:
                    current_time = Time()

                if current_time.set_time(hours, minutes, seconds, time_format):
                    print("Hora asignada correctamente.")
                else:
                    print("Error: Los valores de la hora no están dentro del rango permitido para el formato indicado.")

            except ValueError:
                print("Error de entrada: Las horas, minutos y segundos deben ser números enteros.")

        elif option == "2":
            display_formatted_time(current_time)

        elif option == "3":
            time_str: str = input("Introduzca la cadena de hora (ej. '14:30:00 24 HOURS' o '02:45:30 PM'): ")
            new_time: Optional[Time] = Time.from_string(time_str)
            if new_time is not None:
                current_time = new_time
                print("Hora creada correctamente a partir de la cadena.")

        elif option == "4":
            print(f"Terminando el programa. Objetos Time creados en total: {Time.get_time_count()}.")
            break

        else:
            print("Opción no válida. Por favor, seleccione un número entre 1 y 4.")


if __name__ == "__main__":
    main()